# F Tuần 1 — QLoRA SFT nhẹ cho Qwen2.5-0.5B-Instruct trên lát xLAM 2k (Colab T4)

**Mục tiêu:** dựng được môi trường Colab, chạy trọn một vòng huấn luyện QLoRA bằng script chung
`training/finetune_qlora.py`, chứng minh checkpoint trên Drive cho phép **tiếp tục sau khi mất kết nối**, và
tạo báo cáo dán vào ClickUp.

**Nguyên tắc của notebook này**
- Notebook **chỉ gọi** script trong `docs/contracts/cli.md`; không viết lại logic huấn luyện trong cell.
  Cờ (flag) của script là hợp đồng — muốn đổi phải mở PR riêng.
- Tuần này **chỉ 0.5B**. Cấu hình T4 cố định: `--max-len 2560 --batch-size 4 --grad-accum 4`, fp16,
  gradient checkpointing (trong script), lưu checkpoint mỗi 50 bước lên Drive, `--seed 42`.
  `--max-len 2560` (bội của 256) vì hàng dài nhất của `xlam_2k.train.jsonl` là 2426 token (prompt + completion);
  script **không cắt ngầm** — hàng nào vượt `--max-len` là thoát mã 1, nên dry-run ở Bước 1 phải báo `over_limit=0`.
- Mọi con số trong báo cáo được **đọc từ file** script ghi ra (`log_history.json`, `training_config.json`,
  log), không gõ tay.

**Cần đọc trước:** `docs/contracts/training_row_format.md`, `docs/contracts/cli.md` (mục `finetune_qlora.py`),
`notebooks/finetune/README.md` (lỗi Colab thường gặp).

**Thứ tự chạy:** Cell 1 → 2 → 3 → Bước 1 (dry-run) → Bước 2 (smoke) → Bước 3 (full) → Bước 4 (diễn tập mất
kết nối) → Bước 5 (đồ thị loss) → Bước 6 (sinh một mẫu) → Tạo báo cáo.

### Cell 1 — Thiết lập (ô chung của mọi notebook, copy nguyên văn từ `notebooks/_setup_snippet.md`)
Trên Colab: đọc `GITHUB_TOKEN` từ **Secrets** (biểu tượng chìa khóa ở thanh bên trái, bật *Notebook access*),
`git clone` repo private `thanhhao98/ChatSystem` (bỏ qua nếu đã có) rồi `chdir` vào đó. Trên máy cá nhân: đi lên từ
thư mục hiện tại đến khi gặp `docs/contracts/cli.md`. Ô đặt các biến `REPO`, `GIT_SHA`, `IN_COLAB`, `AUTHOR` và hàm
`run(cmd)`. Kaggle: token đọc từ *Add-ons → Secrets*. **Không bao giờ** in token hay `!cat .git/config` vào output.

In [ ]:
# --- Thiết lập (Colab + local) --------------------------------------------------------------
# Colab : read GITHUB_TOKEN from Secrets, clone the private repo (skip if present), chdir into it.
# Local : walk up from the current directory until the repo root (docs/contracts/cli.md) is found.
# Sets REPO (Path), GIT_SHA, IN_COLAB, AUTHOR and a run() helper that calls repo scripts.
import os, shlex, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
MARKER = "docs/contracts/cli.md"          # exists at the root of every checkout

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _github_token():
    # Colab Secrets -> Kaggle Secrets -> environment variable. Never print the value.
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    return os.environ["GITHUB_TOKEN"]


if (IN_COLAB or IN_KAGGLE) and not Path(MARKER).exists():
    try:
        _token = _github_token()
    except Exception as e:  # SecretNotFoundError / NotebookAccessError / KeyError
        raise RuntimeError(
            "Thiếu secret GITHUB_TOKEN. Colab: biểu tượng chìa khoá (Secrets) -> Add new secret: "
            "Name = GITHUB_TOKEN, Value = Personal access token (classic, scope repo) của tài khoản collaborator "
            "trên thanhhao98/ChatSystem, bật 'Notebook access'. Kaggle: Add-ons -> Secrets -> GITHUB_TOKEN. "
            "Rồi chạy lại ô này.") from e
    if not Path("ChatSystem").exists():
        _r = subprocess.run(["git", "clone", "--quiet", f"https://{_token}@{REPO_HTTPS}", "ChatSystem"],
                            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone thất bại: " + _r.stderr.replace(_token, "<token>"))
    os.chdir("ChatSystem")
    del _token
else:
    _here = Path.cwd().resolve()
    for _cand in [_here, *_here.parents]:
        if (_cand / MARKER).exists():
            os.chdir(_cand)
            break
    else:
        raise FileNotFoundError(f"Không tìm thấy gốc repo (không có {MARKER}) khi đi lên từ {_here}. "
                                "Mở notebook từ bên trong thư mục ChatSystem đã clone.")

REPO = Path.cwd()


def _git(*args):
    # Small helper: run a git command in REPO and return stdout ("" on any failure).
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""


GIT_SHA = _git("rev-parse", "--short", "HEAD") or "no-git"
AUTHOR = os.environ.get("GITHUB_USER") or _git("config", "user.name") or "điền tên"


def run(cmd):
    # Run a repo script (list of args), echo the command, stream its output, return CompletedProcess.
    shown = " ".join(shlex.quote(c) for c in cmd).replace(shlex.quote(sys.executable), "python", 1)
    print("$ " + shown)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    print(f"[exit code = {p.returncode}]")
    return p


print(f"REPO      = {REPO}")
print(f"git HEAD  = {GIT_SHA}")
print(f"python    = {sys.version.split()[0]} · Colab = {IN_COLAB} · Kaggle = {IN_KAGGLE} · author = {AUTHOR}")

### Cell 2 — Cài thư viện theo pins và kiểm tra GPU
Dòng đầu tiên in ra trên Colab miễn phí phải là:

```
Tesla T4 · 15 GB · (7, 5) · USE_BF16=False → fp16
```

- `(7, 5)` là *compute capability*; T4 < 8 nên **fp16**. Script huấn luyện tự chọn dtype theo đúng quy tắc này
  (biến môi trường `FORCE_FP16=1` ép fp16 trên GPU mới hơn để tái lập điều kiện T4).
- Nếu thấy `NO CUDA GPU`: *Runtime → Change runtime type → T4 GPU* rồi chạy lại từ Cell 1.
- Các phiên bản in ra phải trùng `requirements-train.txt`; nếu Colab đổi phiên bản `torch` thì ghi vào comment
  của task và mở issue kèm dòng phiên bản (pins được kiểm tra lại qua PR), **không** tự sửa pins.

In [ ]:
# Cell 2 — install the pinned training stack (Colab/Kaggle only) and probe the GPU.
import os
IN_KAGGLE = bool(globals().get("IN_KAGGLE")) or ((not IN_COLAB) and os.path.isdir("/kaggle/working"))
if (IN_COLAB or IN_KAGGLE) and not Path(MARKER).exists():
    !pip install -q -r requirements-train.txt
else:
    print("Local run: skipping pip install (use your own virtualenv built from requirements-train.txt).")

import importlib

import torch

FORCE_FP16 = os.environ.get("FORCE_FP16") == "1"
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
    GPU_CAP = torch.cuda.get_device_capability(0)
    # dtype is chosen by compute capability, NOT by torch.cuda.is_bf16_supported():
    # the T4 (7,5) only emulates bf16 and the training script would be slow / unstable.
    USE_BF16 = GPU_CAP[0] >= 8 and not FORCE_FP16
    GPU_LINE = (f"{GPU_NAME} · {GPU_GIB:.0f} GB · {GPU_CAP} · USE_BF16={USE_BF16} "
                f"→ {'bf16' if USE_BF16 else 'fp16'}")
else:
    GPU_NAME, GPU_GIB, GPU_CAP, USE_BF16 = None, 0.0, None, False
    GPU_LINE = "NO CUDA GPU (Colab: Runtime → Change runtime type → T4 GPU)"
print(GPU_LINE)

VERSIONS = {}
for _mod in ("torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate"):
    try:
        VERSIONS[_mod] = importlib.import_module(_mod).__version__
    except Exception as exc:  # ImportError, or a broken CUDA extension on CPU-only hosts
        VERSIONS[_mod] = f"not installed ({type(exc).__name__})"
    print(f"{_mod:<13} {VERSIONS[_mod]}")
VERSIONS_LINE = " · ".join(f"{k} {v}" for k, v in VERSIONS.items())

### Cell 3 — Nơi lưu kết quả
Colab tắt phiên bất kỳ lúc nào (idle ~90 phút, hết quota ngày) và **xóa toàn bộ đĩa VM**. Vì vậy mọi kết quả
(checkpoint, adapter, log) ghi thẳng lên Google Drive: `OUT = /content/drive/MyDrive/ChatSystem/runs/<RUN_NAME>`.
Kaggle: `/kaggle/working/runs/<RUN_NAME>`; máy cá nhân: `runs/<RUN_NAME>` trong repo (đã gitignore).
Cache model Hugging Face để trên đĩa VM (mặc định) — không cần đưa lên Drive.

In [ ]:
# Cell 3 — where run outputs go. Colab: Google Drive (survives a disconnect); Kaggle: /kaggle/working; local: runs/.
RUN_NAME = "xlam2k_qwen05b"   # one folder per experiment; keep the SAME name across notebooks 01/02/03

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/ChatSystem/runs/{RUN_NAME}"
elif IN_KAGGLE:
    OUT = f"/kaggle/working/runs/{RUN_NAME}"
else:
    OUT = str(REPO / "runs" / RUN_NAME)   # runs/ is gitignored
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT   # `!python … $OUT/…` lines and subprocesses see the same path

print("OUT =", OUT)

## Xem toàn bộ cờ của script huấn luyện
Đối chiếu với `docs/contracts/cli.md`. Nếu `--help` và `cli.md` khác nhau → đó là lỗi hợp đồng: ghi vào comment của task và mở issue.

In [ ]:
!python training/finetune_qlora.py --help

## Bước 1 — Dry-run: kiểm tra dữ liệu và cách dựng prompt (không cần GPU)
`--dry-run` dựng hàng 0 bằng `tokenizer.apply_chat_template(messages[:-1], tools=<tools>, add_generation_prompt=True)`,
in prompt ra màn hình, đếm số token completion **không bị mask** và chạy *truncation guard* trên toàn bộ tập
(đếm hàng dài hơn `--max-len`). Không tải model, chỉ tải tokenizer (`Qwen/Qwen2.5-0.5B-Instruct`, vài giây).
Log được `tee` vào `$OUT/dry_run.log` để cell báo cáo đọc lại.

In [ ]:
!python training/finetune_qlora.py --dry-run --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl --max-len 2560 2>&1 | tee $OUT/dry_run.log

**Phải thấy ba điều sau, thiếu một là chưa qua:**
1. Prompt in ra có khối `<tools>` … `</tools>` và dòng `assert '<tools>' in prompt: OK` (danh sách tool được render
   bởi chat template, **không** nằm trong text của message — đây là điểm "parity" giữa huấn luyện và phục vụ).
2. Dòng `unmasked completion tokens: N` với **N > 0** (nếu N = 0 thì loss chỉ học trên toàn mask → loss = 0, mô hình
   không học gì — đúng sự cố đã xảy ra ở hệ thống tham chiếu (POC v1)).
3. Dòng `[data] rows=1600 max_len=2560 … over_limit=0`, dòng `[val] rows=200 … over_limit=0` và dòng cuối
   `finetune_qlora --dry-run: OK … over_limit=0`. Với dữ liệu hiện tại trong repo, `max` của tập train là 2426 token
   và của tập val là 1276 — đều dưới 2560.

**Nếu `over_limit > 0` (dry-run in `FAIL` và liệt kê id + số token):** dữ liệu trong repo đã đổi so với lúc viết notebook.
Script **sẽ thoát mã 1** khi huấn luyện thật (không cắt ngầm), nên đừng chạy Bước 2/3 vội. Cách xử lý đúng:
- **Báo ngay** trong bình luận ClickUp: danh sách id và số token (copy từ log) kèm `git HEAD` in ở Cell 1. Nhóm Dữ liệu
  quyết định lọc hay giữ; việc nâng `--max-len` được quyết định trong issue đó (thay đổi cấu hình T4 cố định đi qua PR).
- Nếu issue kết luận nâng `--max-len`: dùng giá trị dry-run gợi ý ở dòng `all rows would fit at --max-len N` (làm tròn lên
  bội của 256) **ở tất cả các bước** của notebook và ghi rõ trong báo cáo — đây là thay đổi cấu hình T4 cố định, phải nói ra.
- **Không** dùng `--max-rows` để "bỏ" hàng: cờ đó chỉ lấy N hàng đầu, không lọc theo độ dài, và làm số liệu không
  so sánh được với người khác. **Không** hạ `--max-len` để "cho qua" — script sẽ từ chối, và cắt ngầm là thứ ta cố tránh.

## Theo dõi VRAM khi huấn luyện
Huấn luyện chạy trong **tiến trình con** (`!python …`), nên `torch.cuda.max_memory_allocated()` trong kernel này
không nhìn thấy nó. Script tự ghi `resolved.peak_vram_gb` (theo `max_memory_allocated`) vào `training_config.json`
khi chạy xong; hai helper dưới đây là **dự phòng**: lấy mẫu `nvidia-smi` mỗi 5 giây để vẫn có VRAM đỉnh nếu lượt chạy
bị ngắt giữa chừng. Cell này chỉ định nghĩa hàm.

In [ ]:
# VRAM monitor helpers: sample `nvidia-smi memory.used` (MiB) into a log while a training subprocess runs.
import shlex
import shutil
import subprocess

_VRAM_PROC = None


def start_vram_monitor(log_path, every_s=5):
    """Start a background sampler writing one MiB value per line to log_path."""
    global _VRAM_PROC
    stop_vram_monitor()
    if not shutil.which("nvidia-smi"):
        print("nvidia-smi not found; VRAM monitor disabled")
        return
    cmd = (f"while true; do nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits "
           f">> {shlex.quote(log_path)}; sleep {every_s}; done")
    _VRAM_PROC = subprocess.Popen(["bash", "-c", cmd])
    print(f"VRAM monitor → {log_path}")


def stop_vram_monitor():
    global _VRAM_PROC
    if _VRAM_PROC is not None:
        _VRAM_PROC.terminate()
        try:
            _VRAM_PROC.wait(timeout=10)
        except subprocess.TimeoutExpired:
            _VRAM_PROC.kill()
        _VRAM_PROC = None


def peak_vram_gib(log_path):
    """Max sampled memory.used in GiB, or None when the log is missing/empty."""
    try:
        with open(log_path) as f:
            vals = [float(x) for x in f.read().split() if x.strip()]
    except FileNotFoundError:
        return None
    return max(vals) / 1024 if vals else None


print("helpers ready: start_vram_monitor / stop_vram_monitor / peak_vram_gib")

## Bước 2 — Smoke test: 200 hàng, 1 epoch (mục tiêu ≈ 3 phút)
Mục đích là bắt lỗi môi trường (OOM, thiếu thư viện, dtype) trước khi tốn ~1 giờ cho lượt đầy đủ.
200 hàng / (4 × 4) = 13 bước. Kết quả vào `$OUT/smoke`. Dòng cuối phải là `finetune_qlora: DONE steps=13 …`.

In [ ]:
start_vram_monitor(f"{OUT}/vram_smoke.log")
!python -u training/finetune_qlora.py --data data/public/xlam_2k.train.jsonl --max-rows 200 --epochs 1 --batch-size 4 --grad-accum 4 --max-len 2560 --save-steps 50 --seed 42 --output-dir $OUT/smoke 2>&1 | tee $OUT/train_smoke.log
stop_vram_monitor()
print("smoke peak VRAM (nvidia-smi):", peak_vram_gib(f"{OUT}/vram_smoke.log"), "GiB")

Kiểm tra: loss giảm qua các bước, `$OUT/smoke/adapter_model.safetensors` tồn tại, không có `CUDA out of memory`.
Nếu OOM: giảm `--batch-size 2` và tăng `--grad-accum 8` (giữ effective batch = 16) — ghi rõ trong báo cáo.
Không hạ `--max-len` để tránh OOM: với dữ liệu hiện tại, mọi giá trị dưới 2426 làm script thoát mã 1 (Bước 1).

## Bước 3 — Lượt đầy đủ: 1.6k hàng × 3 epoch
1600 hàng / 16 = **100 bước mỗi epoch → 300 bước**, checkpoint mỗi 50 bước: `checkpoint-50`, `checkpoint-100`, …
Có `--val` nên script tính `eval_loss` **mỗi 50 bước** (cùng nhịp `--save-steps`) — đường thứ hai trên đồ thị loss.
Thời gian tham khảo trên T4: điền sau lần chạy kiểm chứng trên Colab T4 (xem `README.md`). Trong lúc chạy, mở tab Drive và
quan sát thư mục `ChatSystem/runs/<RUN_NAME>/full/` — Bước 4 cần bạn thấy `checkpoint-100/` xuất hiện.

In [ ]:
start_vram_monitor(f"{OUT}/vram_full.log")
!python -u training/finetune_qlora.py --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl --epochs 3 --batch-size 4 --grad-accum 4 --max-len 2560 --save-steps 50 --logging-steps 10 --seed 42 --output-dir $OUT/full 2>&1 | tee $OUT/train_full.log
stop_vram_monitor()
print("full peak VRAM (nvidia-smi):", peak_vram_gib(f"{OUT}/vram_full.log"), "GiB")

## Bước 4 — Diễn tập mất kết nối (bắt buộc)
Colab **sẽ** ngắt bạn giữa một lượt huấn luyện thật. Diễn tập để chắc rằng bạn khôi phục được:

1. Chạy Bước 3. Mở Google Drive trên trình duyệt, vào `ChatSystem/runs/<RUN_NAME>/full/`.
2. Khi thấy `checkpoint-100/` có đủ `adapter_model.safetensors`, `optimizer.pt`, `trainer_state.json`
   (Drive đồng bộ trễ vài chục giây), chọn **Runtime → Disconnect and delete runtime**. Lượt huấn luyện chết.
3. Kết nối lại GPU, chạy lại **Cell 1 → 2 → 3** và cell helper VRAM (đĩa VM đã mất, Drive còn nguyên).
4. Chạy cell dưới đây: cùng cờ với Bước 3, thêm `--resume-from-checkpoint $OUT/full/checkpoint-100`.
5. Trong log phải thấy Trainer **tiếp tục từ bước 100**: dòng `Resuming from …/checkpoint-100`, dòng của
   Trainer `Continuing training from checkpoint … global step 100`, thanh tiến trình bắt đầu ở `100/300`, và dòng log
   loss đầu tiên sau đó có `step` > 100 (với `--logging-steps 10` là bước 110; muốn thấy đúng "step 101" thì thêm
   `--logging-steps 1` cho lượt resume).

Nếu lỡ để lượt Bước 3 chạy xong rồi mới đọc mục này: vẫn phải diễn tập — chạy cell dưới đây với `checkpoint-100`,
Trainer sẽ tiếp tục từ 100 → 300 (tốn thêm ~2/3 thời gian), kết quả cuối tương đương.

In [ ]:
start_vram_monitor(f"{OUT}/vram_resume.log")
!python -u training/finetune_qlora.py --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl --epochs 3 --batch-size 4 --grad-accum 4 --max-len 2560 --save-steps 50 --logging-steps 10 --seed 42 --output-dir $OUT/full --resume-from-checkpoint $OUT/full/checkpoint-250 2>&1 | tee $OUT/resume.log
stop_vram_monitor()

In [ ]:
# Show the evidence that training resumed after step 100 (read from the tee'd log).
import re

try:
    with open(f"{OUT}/resume.log", errors="replace") as f:
        lines = [ln.rstrip() for ln in f]
except FileNotFoundError:
    lines = []
hits = [ln for ln in lines if re.search(r"resuming from|continuing training|global step", ln, re.I)]
steps = [int(m.group(1)) for ln in lines for m in [re.search(r"'step':\s*(\d+)", ln)] if m]
print("resume evidence:", *hits[:3], sep="\n  ")
print("first logged step after resume:", min(steps) if steps else "n/a (chưa chạy Bước 4)")

## Bước 5 — Đồ thị loss từ `log_history.json`
Script ghi `trainer.state.log_history` ra `$OUT/full/log_history.json`. Sau khi resume, file này chứa **cả** phần
trước và sau checkpoint-100 (Trainer nạp lại `trainer_state.json`), nên đường loss liền mạch.
Kỳ vọng: train loss giảm mạnh trong ~30 bước đầu rồi phẳng dần; `eval_loss` không tăng lại ở epoch 3
(nếu tăng → overfit nhẹ, ghi nhận trong báo cáo, không cần sửa tuần này).

In [ ]:
# Plot train loss (and eval loss when present) from log_history.json; save a PNG next to the adapter.
import json

import matplotlib.pyplot as plt

LOG_HISTORY = f"{OUT}/full/log_history.json"
try:
    with open(LOG_HISTORY) as f:
        hist = json.load(f)
    if isinstance(hist, dict):
        hist = hist.get("log_history", [])
except FileNotFoundError:
    hist = []

train = [(h["step"], h["loss"]) for h in hist if "loss" in h and "step" in h]
evals = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h and "step" in h]
if not train:
    print(f"Chưa có dữ liệu loss ({LOG_HISTORY} không tồn tại hoặc rỗng) — chạy Bước 3 trước.")
else:
    fig, ax = plt.subplots(figsize=(7, 3.6), dpi=110)
    ax.plot(*zip(*train), color="#2a78d6", linewidth=2, label="train loss")
    ax.annotate(f"train {train[-1][1]:.3f}", xy=train[-1], xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9, color="#333")
    if evals:
        ax.plot(*zip(*evals), color="#eb6834", linewidth=2, marker="o", markersize=5, label="eval loss")
        ax.annotate(f"eval {evals[-1][1]:.3f}", xy=evals[-1], xytext=(6, 0), textcoords="offset points",
                    va="center", fontsize=9, color="#333")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("loss")
    ax.set_title(f"QLoRA SFT loss — {RUN_NAME} (log_history.json)", fontsize=10, loc="left")
    ax.grid(True, color="#e6e6e6", linewidth=0.6)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    if evals:
        ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(f"{OUT}/full/loss_curve.png", bbox_inches="tight")
    plt.show()
    print(f"steps logged: {len(train)} · final train loss {train[-1][1]:.4f}"
          + (f" · final eval loss {evals[-1][1]:.4f}" if evals else ""))
    print("saved:", f"{OUT}/full/loss_curve.png")

## Bước 6 — Sinh một mẫu greedy bằng adapter vừa huấn luyện
Gọi `training/predict_toolcall.py` (backend `transformers`, greedy `--temperature 0` mặc định) trên **một** bản ghi
của bộ eval công khai (`--limit 1`). Script tự nạp base fp16 + adapter, render tools bằng `apply_chat_template`,
và ghi `raw_text` + `predicted_tool_calls`. Đây chỉ là kiểm tra "mô hình có nói đúng định dạng `<tool_call>`
không"; **độ chính xác** đo ở notebook 02.

In [ ]:
!python training/predict_toolcall.py --backend transformers --model-path Qwen/Qwen2.5-0.5B-Instruct --adapter $OUT/full --eval data/public/xlam_2k.eval.json --limit 1 --out /tmp/one.jsonl

In [ ]:
# Print the single prediction: header line first, then one record per eval item.
import json

ONE_RAW_TEXT, ONE_RECORD = None, None
try:
    with open("/tmp/one.jsonl") as f:
        rows = [json.loads(ln) for ln in f if ln.strip()]
    header = next((r for r in rows if r.get("header")), {})
    ONE_RECORD = next((r for r in rows if not r.get("header")), None)
    print("header:", json.dumps({k: header.get(k) for k in ("model", "date", "git_sha", "sha256_eval")}, ensure_ascii=False))
    if ONE_RECORD:
        ONE_RAW_TEXT = ONE_RECORD.get("raw_text")
        print("id:", ONE_RECORD.get("id"), "| predicted_tool:", ONE_RECORD.get("predicted_tool"),
              "| latency_ms:", ONE_RECORD.get("latency_ms"))
        print("raw_text:\n", ONE_RAW_TEXT)
        print("predicted_tool_calls:", json.dumps(ONE_RECORD.get("predicted_tool_calls"), ensure_ascii=False))
except FileNotFoundError:
    print("/tmp/one.jsonl chưa có — chạy cell Bước 6 trước (cần GPU).")

## Tạo báo cáo
Cell dưới in đúng khối văn bản để **dán vào bình luận** của task ClickUp *F Tuần 1*. Mọi số đều được đọc từ file
script ghi ra; mục nào chưa có hiện `n/a` — không tự điền tay. Kèm theo ảnh `full/loss_curve.png` (tải từ Drive).

In [ ]:
# "## Tạo báo cáo" — prints ONE Markdown block to paste as the ClickUp task comment.
# Every number is read from files the scripts wrote; anything missing prints as n/a (never typed by hand).
import datetime
import json
import re
import subprocess


def _read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _grep(path, pattern, max_lines=3):
    """Last `max_lines` lines of `path` matching `pattern` (case-insensitive); [] when the file is missing."""
    try:
        with open(path, errors="replace") as f:
            hits = [ln.rstrip() for ln in f if re.search(pattern, ln, re.I)]
        return hits[-max_lines:]
    except FileNotFoundError:
        return []


def _ls(path):
    r = subprocess.run(["ls", "-la", path], capture_output=True, text=True)
    return r.stdout.strip() or f"(missing: {path})"


def _log_history(path):
    """log_history.json is trainer.state.log_history: a list of dicts (accept {"log_history": [...]} too)."""
    hist = _read_json(path)
    if isinstance(hist, dict):
        hist = hist.get("log_history", [])
    return hist or []


ENV_LINE = "Colab" if IN_COLAB else ("Kaggle" if globals().get("IN_KAGGLE") else "local")
TODAY = datetime.date.today().isoformat()


TASK = "Tuần 1 — Dựng môi trường Colab và chạy QLoRA SFT nhẹ (Qwen2.5-0.5B-Instruct) trên lát xLAM 2k"
FULL = f"{OUT}/full"
SMOKE = f"{OUT}/smoke"
ONE_RECORD = globals().get("ONE_RECORD")                       # set by the Step 6 read cell (may be skipped)
_peak_vram = globals().get("peak_vram_gib", lambda _p: None)   # set by the VRAM helper cell (may be skipped)
cfg = _read_json(f"{FULL}/training_config.json") or {}
resolved = cfg.get("resolved") or {}
hist_full = _log_history(f"{FULL}/log_history.json")
hist_smoke = _log_history(f"{SMOKE}/log_history.json")


def _summary(hist):
    """(train_runtime_s, final_train_loss, final_eval_loss, last_step) from a trainer log history."""
    fin = next((h for h in reversed(hist) if "train_runtime" in h), {})
    losses = [h for h in hist if "loss" in h and "step" in h]
    evals = [h for h in hist if "eval_loss" in h]
    return (fin.get("train_runtime"), losses[-1]["loss"] if losses else None,
            evals[-1]["eval_loss"] if evals else None, losses[-1]["step"] if losses else None)


rt_full, loss_full, eval_full, step_full = _summary(hist_full)
rt_smoke, loss_smoke, _, step_smoke = _summary(hist_smoke)

# Peak VRAM: (1) written by the script (max_memory_allocated inside the training process),
# (2) the nvidia-smi sampler, (3) otherwise tell the reader how to obtain it.
if cfg.get("peak_vram_gib") is not None:
    vram_line = f"{float(cfg['peak_vram_gib']):.2f} GiB (training_config.json)"
elif resolved.get("peak_vram_gb") is not None:
    vram_line = f"{float(resolved['peak_vram_gb']):.2f} GB (training_config.json → resolved.peak_vram_gb, torch.cuda.max_memory_allocated)"
elif _peak_vram(f"{OUT}/vram_full.log") is not None:
    vram_line = f"{_peak_vram(f'{OUT}/vram_full.log'):.2f} GiB (nvidia-smi sampler, memory.used)"
else:
    vram_line = "n/a — chạy `nvidia-smi` trong lúc huấn luyện và ghi lại giá trị memory.used lớn nhất"

dry_summary = _grep(f"{OUT}/dry_run.log", r"finetune_qlora --dry-run:", 1) or ["n/a (chưa chạy Bước 1)"]
dry_unmasked = _grep(f"{OUT}/dry_run.log", r"unmasked completion tokens", 1) or ["n/a"]
dry_over = _grep(f"{OUT}/dry_run.log", r"^\[(data|train)\] rows=", 1) or ["n/a"]
dry_ids = _grep(f"{OUT}/dry_run.log", r"^\s+pub-[0-9a-f]+:\s+\d+", 10)
done_full = _grep(f"{OUT}/train_full.log", r"finetune_qlora: DONE", 1) or _grep(f"{OUT}/resume.log", r"finetune_qlora: DONE", 1) or ["n/a"]
resume_hits = _grep(f"{OUT}/resume.log", r"resuming from|continuing training|global step", 2) or ["n/a (chưa chạy Bước 4)"]
n_train = resolved.get("n_train")
m = re.search(r"rows=(\d+).*over_limit=(\d+)", dry_summary[0])
trunc_pct = (f"{100.0 * int(m.group(2)) / int(m.group(1)):.1f}% ({m.group(2)}/{m.group(1)}) — kỳ vọng 0"
             if m else "n/a")
_fmt = lambda v, f="{:.4f}": (f.format(v) if isinstance(v, (int, float)) else "n/a")

CMDS = [
    "python training/finetune_qlora.py --dry-run --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl --max-len 2560",
    f"python training/finetune_qlora.py --data data/public/xlam_2k.train.jsonl --max-rows 200 --epochs 1 --batch-size 4 --grad-accum 4 --max-len 2560 --save-steps 50 --seed 42 --output-dir {OUT}/smoke",
    f"python training/finetune_qlora.py --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl --epochs 3 --batch-size 4 --grad-accum 4 --max-len 2560 --save-steps 50 --logging-steps 10 --seed 42 --output-dir {OUT}/full",
    f"python training/finetune_qlora.py … (cùng cờ) --resume-from-checkpoint {OUT}/full/checkpoint-250",
    f"python training/predict_toolcall.py --backend transformers --model-path Qwen/Qwen2.5-0.5B-Instruct --adapter {OUT}/full --eval data/public/xlam_2k.eval.json --limit 1 --out /tmp/one.jsonl",
]

lines = [
    f"## Báo cáo {TASK} — {TODAY} — {AUTHOR}",
    f"- Notebook: `notebooks/finetune/01_qlora_sft_colab.ipynb` @ `{GIT_SHA}` · môi trường: {ENV_LINE} · RUN_NAME `{RUN_NAME}` · OUT `{OUT}`",
    f"- GPU: {GPU_LINE}",
    f"- Phiên bản: {VERSIONS_LINE}",
    f"- Dry-run: `{dry_unmasked[0].strip()}` · `{dry_over[0].strip()}` · `{dry_summary[0].strip()}`",
    f"- Hàng bị cắt / bỏ: {trunc_pct}" + (f" · id vượt: {', '.join(x.strip() for x in dry_ids)}" if dry_ids else ""),
    f"- Smoke (200 hàng, 1 epoch): train_runtime = {_fmt(rt_smoke, '{:.0f} s')} · final loss = {_fmt(loss_smoke)} · steps = {step_smoke or 'n/a'}",
    f"- Full (1.6k hàng × 3 epoch, seed 42): train_runtime = {_fmt(rt_full, '{:.0f} s')} ({_fmt(rt_full / 60 if rt_full else None, '{:.1f} phút')}) · final train loss = {_fmt(loss_full)} · final eval loss = {_fmt(eval_full)} · steps = {step_full or 'n/a'} · n_train = {n_train or 'n/a'} · dtype = {resolved.get('dtype', 'n/a')}",
    f"- Dòng DONE của script: `{done_full[0].strip()}`",
    f"- Peak VRAM: {vram_line}",
    f"- Resume (Bước 4): " + " | ".join(f"`{h.strip()}`" for h in resume_hits),
    f"- Mẫu greedy (Bước 6): id = {ONE_RECORD.get('id') if ONE_RECORD else 'n/a'} · predicted_tool = {ONE_RECORD.get('predicted_tool') if ONE_RECORD else 'n/a'}",
    f"- Adapter (`ls -la {FULL}`):",
    "```", _ls(FULL), "```",
    "- Lệnh đã chạy:",
    "```", *CMDS, "```",
    "- Đồ thị loss: đính kèm `full/loss_curve.png` từ Drive.",
]
print("\n".join(lines))